1.1 修改列名

In [ ]:
import pandas as pd

file_name = 'comment_test.csv'
# 1. 读入
df = pd.read_csv(f'/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/{file_name}', dtype=str)
# 2. 用一个字典指定 “原列名: 新列名”
rename_dict = {
    'ML2_proxy3b_probability': 'ML2_proxy3f_probability'
}
# 3. 执行重命名
df.rename(columns=rename_dict, inplace=True)
# 4. 保存（覆盖原文件或另存为新文件）
df.to_csv(f'/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/{file_name}', index=False)

1.2 删除某一列的数据

In [ ]:
import pandas as pd
csv_path = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_test.csv"
df = pd.read_csv(csv_path)

# 假设要删除的列名是 'col_to_remove'
delete_col = 'ML1_proxy_9_probability'
df.drop(columns=[delete_col], inplace=True)

# 将修改后的 DataFrame 写回文件（覆盖原文件）
df.to_csv(csv_path, index=False)

print(f"已删除列 {delete_col} 并保存至 {csv_path}")


1.3 输出csv文件所有的属性列

In [ ]:
import pandas as pd

# 读取 CSV（如果有中文路径或编码问题，可加 encoding 参数）
# file_name = 'comment_cleaned2.csv'
# file_name = 'post.csv'
file_name = 'comment.csv'
df = pd.read_csv(f'/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/{file_name}', encoding='utf-8')
# 输出所有列名
print(df.columns.tolist())


1.4 删除文件某一列

In [ ]:
import pandas as pd

def delete_csv_column(input_csv: str, output_csv: str, column_name: str) -> None:
    """
    从 input_csv 中删除名为 column_name 的列，并将结果保存为 output_csv。
    Args:
        input_csv (str): 原始 CSV 文件路径。
        output_csv (str): 删除列后输出的 CSV 文件路径。
        column_name (str): 要删除的列名。
    Raises:
        KeyError: 如果 column_name 不在 CSV 的列中。
    """
    # 1. 读取 CSV
    df = pd.read_csv(input_csv)
    # 2. 检查列是否存在
    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found in {input_csv}")
    # 3. 删除列
    df.drop(columns=[column_name], inplace=True)
    # 4. 保存结果
    df.to_csv(output_csv, index=False)
    print(f"Deleted column '{column_name}' and saved to '{output_csv}'")
delete_csv_column(
    input_csv="/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment.csv",
    output_csv="/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment.csv",
    column_name="ML2_score"
)

1.5 读取post.csv文件，只保留'id:ID', ':LABEL', 'comments', 'body', 'createdAt', 'creator', 'score', 'sensitive', 'upvotes', 'username', 'pc_count', 'ML1_proxy1_probability', 'ML1_proxy2_probability'...属性到 post1.csv

In [ ]:
import pandas as pd

# 要保留的列列表
cols_to_keep = [
    'id:ID',
    ':LABEL',
    'comments',
    'body',
    'createdAt',
    'creator',
    'score',
    'sensitive',
    'upvotes',
    'username',
    'pc_count',
    'ML1_proxy1_probability',
    'ML1_proxy2_probability',
    'ML1_proxy3_probability',
    'ML1_proxy4_probability',
    'ML1_proxy5_probability',
    'ML1_oracle1_probability',
    'ML1_oracle2_probability',
    'ML1_oracle3_probability',
]

# 读取 CSV，仅加载指定列
df = pd.read_csv('/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post.csv', usecols=cols_to_keep)

# 保存到新的 CSV 文件
df.to_csv('/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post1.csv', index=False)

print("已成功将指定列写入 post1.csv")


In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 进度条库

# 目标 ID
target_id = "c7645180a7d7411689174365a29fbf30"

# 文件夹路径
folder = "valid_user"

# 获取所有 CSV 文件名
csv_files = [f for f in os.listdir(folder) if f.endswith(".csv")]

# 遍历并显示进度
for filename in tqdm(csv_files, desc="Searching CSV files"):
    filepath = os.path.join(folder, filename)
    try:
        df = pd.read_csv(filepath)
        if 'id' in df.columns and target_id in df['id'].astype(str).values:
            print(f"\n✅ Found in: {filename}")
            break
    except Exception as e:
        print(f"\n⚠️ Error reading {filename}: {e}")


批量处理所有 CSV 文件中的乱码

In [6]:
import pandas as pd
import re

def clean_csv_text_column(
    input_path: str,
    column_name: str = 'bio',
    output_path: str = None,
    encoding_list: list = ['utf-8', 'utf-8-sig', 'ISO-8859-1', 'cp1252']
):
    """
    清洗单个 CSV 文件中指定列的乱码字符，并保存为新文件。

    参数:
    - input_path: 原始 CSV 文件路径
    - column_name: 需要清洗的列名（默认是 'body'）
    - output_path: 清洗后保存的文件路径（默认自动生成）
    - encoding_list: 尝试使用的编码列表，依次尝试直到成功读取文件
    """
    def clean_text(text):
        if isinstance(text, str):
            return re.sub(r'[^\x00-\x7F]+', '', text)  # 去除非ASCII字符
        return text

    df = None
    for enc in encoding_list:
        try:
            df = pd.read_csv(input_path, encoding=enc)
            break
        except Exception:
            continue

    if df is None:
        raise ValueError(f"无法读取文件：{input_path}，尝试的编码失败：{encoding_list}")

    if column_name not in df.columns:
        raise ValueError(f"列 '{column_name}' 不存在于文件中。")

    df[column_name] = df[column_name].apply(clean_text)

    if output_path is None:
        output_path = input_path.replace('.csv', '_cleaned3.csv')

    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ 清洗完成，保存为: {output_path}")
clean_csv_text_column("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/user_cleaned2.csv", column_name="bio")

✅ 清洗完成，保存为: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/user_cleaned2_cleaned3.csv


2.将csv某一列（字符串）的所有属性值变成小写字母

In [ ]:
import pandas as pd
# 1. 读取 CSV
in_csv  = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_ML2_proxy2.csv"
out_csv = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_ML2_proxy2.csv"
df = pd.read_csv(in_csv)

# 2. 将指定列转为小写（假设列名为 'your_column'）
df['ML2_label'] = df['ML2_label'].str.lower()
# 3. 保存结果
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"已将 'your_column' 转为小写，并保存到：{out_csv}")


3.将post_LLM_filtered.csv文件中， LLM_label列, 并将列内的yes改为deepseek_yes, no改为deepseek_no

In [ ]:
import pandas as pd

# 1. 读取 CSV（包含所有列）
input_path = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_test.csv'
df = pd.read_csv(input_path)

# 2. 替换 ML2_oracle1_label 列中的 yes/no/neutral
df['ML2_oracle1_label'].replace({
    'positive': 'positive',
    'negative': 'negative',
    'neutral':  'neutral'
}, inplace=True)

# 3. （可选）检查一下前几行
print(df[['ML2_oracle1_label']].tail())

# 4. 统计 ML2_oracle1_label 下每个值的元组数
counts = df['ML2_oracle1_label'].value_counts()
print("各类别数量：")
print(counts)

# 5. 保存回 CSV（覆盖原文件或另存为新文件）
output_path = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_test.csv'
df.to_csv(output_path, index=False)
print(f"已保存为：{output_path}")


4.查找post.csv文件中body字段为'It's official. Wife and I have voted. In person.'  的元组信息

In [ ]:
import pandas as pd

# 1. 读取 CSV（假设包含所有列，且 body 列名正好是 'body'）
df = pd.read_csv('/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_ML.csv')

# 2. 精确匹配 body 字段
target = "It's official. Wife and I have voted. In person."
df_match = df[df['body'] == target]

# 3. 输出匹配到的行
if df_match.empty:
    print("未找到匹配的记录。")
else:
    print("找到以下记录：")
    print(df_match.to_string(index=False))


5.已知两个文件，post_ML.csv文件post_LLM_cleaned_1.csv文件，两个文件的元组通过id:ID属性字段一一对应，使用post_ML.csv文件中的ML1_probability 和 ML1_label 更新在post_LLM_cleaned_1.csv文件中对应元组的属性

In [ ]:
import pandas as pd

file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
# 读取
df_ml  = pd.read_csv(file_dir + 'post_ML.csv').set_index('id:ID')
df_llm = pd.read_csv(file_dir + 'post_LLM_cleaned_1.csv').set_index('id:ID')

# 直接从 df_ml 对应索引取列，赋给 df_llm
df_llm['ML1_probability'] = df_ml['ML1_probability']
df_llm['ML1_label']       = df_ml['ML1_label']

# 如果需要把索引重置回列
df_llm.reset_index(inplace=True)

# 保存
df_llm.to_csv(file_dir + 'post_LLM_cleaned_2.csv', index=False)
print("更新完成，输出文件：post_LLM_cleaned_2.csv")


6.排序
6.1 按照 'workload1_probability' 字段大小降序排列

In [ ]:
import pandas as pd

# data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3'
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
csv_file = data_dir + '/comment_test.csv'
sort_column = 'ML2_oracle1_probability'
# 读取 posts.csv 文件
df = pd.read_csv(csv_file)

# 按照 'workload1_probability' 字段大小降序排列
df_sorted = df.sort_values(by=sort_column, ascending=False)


# 保存排序后的数据到新的文件
df_sorted.to_csv(csv_file, index=False)


print(f"文件已按照 {sort_column} 排序并保存为 '{csv_file}'")


6.2 按照ML2_oracle1_label 排序，ML2_oracle1_label  的取值有 positive、negative、neutral ，再在postive这个值内，按照ML2_oracle1_probability

In [ ]:
import pandas as pd

# data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3'
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
csv_file = data_dir + '/comment_test.csv'

# 读取 comment_test.csv
df = pd.read_csv(csv_file)

# 指定 label 的排序顺序
label_order = ['positive2o1', 'negative2o1', 'neutral2o1']
df['ML2_oracle1_label'] = pd.Categorical(
    df['ML2_oracle1_label'],
    categories=label_order,
    ordered=True
)

# 按 label 顺序（positive→negative→neutral）排列，
# 并在同一 label 组内，按概率降序
df_sorted = df.sort_values(
    by=['ML2_oracle1_label', 'ML2_oracle1_probability'],
    ascending=[True, False]
)

# 保存回原文件（或另存为新文件）
df_sorted.to_csv(csv_file, index=False)

print(f"已按 ML2_oracle1_label（{label_order}）及同组内 ML2_oracle1_probability 降序排序，并保存至 {csv_file}")


7.将post_LLM_cleaned_4.csv文件中， ML1_oracle1_probability 和ML1_oracle1_label 两列加入到将post_LLM_cleaned_5.csv文件中

In [ ]:
import pandas as pd

# 1. 读取两个文件
file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
df4 = pd.read_csv(file_dir + 'post_LLM_cleaned_4.csv')
df5 = pd.read_csv(file_dir + 'post_LLM_cleaned_5.csv')

# 2. 直接按行把两列赋值过去
df5[['ML1_oracle1_probability', 'ML1_oracle1_label']] = \
    df4[['ML1_oracle1_probability', 'ML1_oracle1_label']]

# 3. 保存新的文件（会覆盖原来的 post_LLM_cleaned_5.csv）
df5.to_csv(file_dir + 'post_LLM_cleaned_5.csv', index=False)


8.已知post.csv文件， 读取comment.csv，如果该文件中的元组无法在post文件中找到对应的post则删除

In [ ]:
import pandas as pd

# 加载 post.csv，假设 post 的唯一标识字段为 'id'
file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
post_df = pd.read_csv(file_dir + 'post_LLM_cleaned.csv')
valid_post_ids = set(post_df['id:ID'])

# 加载 comment.csv，假设 comment 中关联的 post 字段名为 'post'
comment_df = pd.read_csv(file_dir + 'comment.csv')

# 过滤出 post 字段值在 post_df['id'] 中的行
filtered_comment_df = comment_df[comment_df['post'].isin(valid_post_ids)]

# 保存过滤后的 comment 数据
filtered_comment_df.to_csv(file_dir + 'comment_cleaned.csv', index=False)

print(f"原始评论数: {len(comment_df)}，保留评论数: {len(filtered_comment_df)}")


#### 9. 删除英文单词数 ≤ 1 或 非英文句子的行

In [ ]:
import re
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm import tqdm

# 保证语言检测结果稳定
DetectorFactory.seed = 0

# 进度条集成 pandas
tqdm.pandas(desc="Processing rows")

# 1. 读取 CSV
df = pd.read_csv(
    '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_cleaned.csv',
)

# 2. 定义两类筛选函数
def is_letters_le_2(text: str) -> bool:
    # 统计所有英文字母出现次数
    letters = re.findall(r'[A-Za-z]', str(text))
    return len(letters) <= 1

def is_non_english(text: str) -> bool:
    try:
        return detect(str(text)) != 'en'
    except LangDetectException:
        # 检测失败也视作“非英文”
        return True

# 3. 分别打掩码并计数
mask_le2 = df['body'].progress_apply(is_letters_le_2)
mask_non_en = df['body'].progress_apply(is_non_english)

count_le2 = mask_le2.sum()
count_non_en = mask_non_en.sum()

print(f"字母数 ≤ 1 的行数：{count_le2}")
print(f"非英文句子的行数：{count_non_en}")

# 4. 删除这两类行（逻辑或）
df_clean = df.loc[~(mask_le2 | mask_non_en)].reset_index(drop=True)

# 5. 保存清洗后的结果
output_path = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_cleaned1.csv'
df_clean.to_csv(output_path, index=False)
print(f"\n已删除上述两类行，清洗后数据已保存至：{output_path}")


#### 10. ML2_proxy1_label 为 LABEL_0 行的 ML2_proxy1_probability 属性值更改为 1-ML2_proxy1_probability

In [ ]:
import pandas as pd

# 假设已读取 DataFrame 为 df
file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
file_name = 'comment_test.csv'
df = pd.read_csv(file_dir + file_name)

# 找出 ML2_proxy1_label 为 'LABEL_0' 的行，并将其 ML2_proxy1_probability 更新为 1 - 原值
label = 'LABEL_0'
pro = 'ML2_proxy7_probability'
mask = df['ML2_proxy7_label'] == label
df.loc[mask, pro] = 1 - df.loc[mask, pro]

# 如需保存回 CSV（可覆盖原文件或另存为新文件）
df.to_csv(file_dir+file_name, index=False)


10.1 对文件某一列值同一修改 ：加减乘除

In [ ]:
import pandas as pd

# 1. 读取 CSV
file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
file_name = 'post.csv'
df = pd.read_csv(file_dir + file_name, encoding='utf-8')
# 假设要修改的列名叫 "score"
col = 'ML1_proxy5_probability'
# 2. 批量修改
# 加 10
df[col] = 1- df[col] 

# 3. 保存回文件（可覆盖原文件或另存为新文件）
df.to_csv(file_dir + file_name, index=False, encoding='utf-8')


#### 11. 保留post.csv 文件中的 属性id:ID,:LABEL,comments,body,createdAt,creator,score,sensitive,upvotes,username,pc_count

In [ ]:
import pandas as pd

# 1. 读取原始 CSV
file_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
file_name = 'post_LLM_cleaned.csv'
df = pd.read_csv(file_dir+file_name)

# 2. 指定要保留的列
cols_to_keep = [
    'id:ID', ':LABEL', 'comments', 'body', 'createdAt',
    'creator', 'score', 'sensitive', 'upvotes', 'username', 'pc_count','LLM_label'
]

# 3. 筛选
df_filtered = df[cols_to_keep]

# 4. 保存到新文件（或覆盖原文件）
df_filtered.to_csv(file_dir + 'post_origin.csv', index=False)
print("已保留指定列并保存至 post_origin.csv")


12.合并两个csv文件

为csv文件添加某一列，并设置默认值

In [1]:
import pandas as pd
datadir = "/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data"
file_name = "comment_child.csv"
csv_path = f"{datadir}/{file_name}"

# 读取 CSV
df = pd.read_csv(csv_path)

# 添加新列并设置默认值（例如默认值为 0 或空字符串）
df[":LABEL"] = 'Comment'   # 或者 ""
# 也可以设为布尔或其他类型
# df["flag"] = False

# 保存回原文件（覆盖）
df.to_csv(csv_path, index=False)


合并两个csv文件，相同列合并，不同列默认为空，并输出不同列

In [2]:
import pandas as pd

def merge_csv_files(file1, file2, output_file, fillna_value=""):
    """
    合并两个 CSV 文件：
      - 相同列合并
      - 不同列自动补空
      - 输出合并文件和不同列信息
    """
    # 读取 CSV 文件
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    # 找出所有列名的并集
    all_columns = sorted(set(df1.columns) | set(df2.columns))

    # 统一列名（补空列）
    df1 = df1.reindex(columns=all_columns)
    df2 = df2.reindex(columns=all_columns)

    # 填充缺失值
    df1 = df1.fillna(fillna_value)
    df2 = df2.fillna(fillna_value)

    # 合并（按行拼接）
    merged = pd.concat([df1, df2], ignore_index=True)

    # 找出不同列
    only_in_1 = sorted(set(df1.columns) - set(df2.columns))
    only_in_2 = sorted(set(df2.columns) - set(df1.columns))
    diff_columns = {
        "only_in_file1": only_in_1,
        "only_in_file2": only_in_2
    }

    # 保存结果
    merged.to_csv(output_file, index=False)
    print(f"✅ 合并完成，输出文件：{output_file}")
    print(f"📊 文件1独有列: {only_in_1}")
    print(f"📊 文件2独有列: {only_in_2}")

    return merged, diff_columns


# === 示例调用 ===
datadir = "/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data"
file_name = "comment_child.csv"
file1 = datadir +f"/{file_name}"
file2 = datadir +f"/comment.csv"
output_file = datadir + '/comment_merged.csv'

merged_df, diff_cols = merge_csv_files(file1, file2, output_file)


✅ 合并完成，输出文件：/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data/comment_merged.csv
📊 文件1独有列: []
📊 文件2独有列: []
